In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="CalculateMoreVariables", dataName="UpdraftArea",
                                dtype='float32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#Variable Loading Functions

def GetVarNames():
    return ['A_g','A_c','winterp']

def GetInputVariables(inputDataDirectory, timeString, varNames):
    inputDictionary = {varName: CallVariable(ModelData, DataManager, timeString, varName) for varName in varNames}
    return inputDictionary

In [ ]:
#Updraft Area Calculation Functions

def CalculateBinaryArea_V1(A,w, shapeApproximation="oval"): 
    """
    For each updraft point, calculate area as the true grid cell count
    of the 2D connected region it belongs to.

    RECOMMENDED OVER V3, SINCE FASTER AND MORE INTUITIVE AREA ESTIMATE
    """
    [Nz, Ny, Nx] = A.shape
    updraftArea = np.full(A.shape, np.nan)

    for k in range(Nz):
        slice2D = A[k, :, :]
        if slice2D.sum() == 0: continue

        # labeled, n_features = label(slice2D)
        labeled, n_features = GetLabeled_2D(slice2D,varSlice2D=w[k])
        
        for i in range(1, n_features + 1):
            mask = labeled == i
            updraftArea[k][mask] = mask.sum()  # grid cell count, not x*y

    updraftArea *= ModelData.dx * ModelData.dy
    if shapeApproximation == "oval":
        updraftArea *= (np.pi / 4)

    return updraftArea, [], []

def LabelSizesOfIndividualUpdrafts(array1D):
    """Label connected regions and return per-cell sizes."""
    labeled, _ = label(array1D)
    sizes = np.bincount(labeled.ravel())
    return labeled, sizes

from scipy.ndimage import maximum_filter
from skimage.segmentation import watershed
def GetLabeled_2D(slice2D,varSlice2D):
    """
    Instead of labeled, n_features = label(slice2D)
    Uses watershedding algorithm to label 2d objects
    """
    mask = slice2D
    
    # find local maxima of varSlice2D within the updraft mask
    local_max = (varSlice2D == maximum_filter(varSlice2D, size=5*ModelData.kms)) & mask
    markers, _ = label(local_max)
    
    # watershed splits the connected mask using varSlice2D as the "elevation" to climb down from peaks
    labeled = watershed(-varSlice2D, markers=markers, mask=mask)
    n_features = labeled.max()
    
    return labeled, n_features

def ApplyLengths(array1D, sizes,labeled):
    return np.where(array1D, sizes[labeled], np.nan)    

from scipy.ndimage import label
def CalculateBinaryLength(A, dimension, BC_x='open', BC_y='periodic'):
    """
    For each updraft point, calculate the length of the contiguous 
    updraft run it belongs to, along the specified dimension.
    """

    #Get Dimensional Information
    [Nz, Ny, Nx] = A.shape; length = np.full(A.shape, np.nan)
    [N, N_loop, BC] = (Nx, Ny, BC_x) if dimension == 'x' else (Ny, Nx, BC_y)

    #Looping
    for k in range(Nz): #Looping over all levels
        for n in range(N_loop): #Looping over each dimensional slice
            array1D = A[k, n, :] if dimension == 'x' else A[k, :, n] #Indexing corresponding 1d array
            if array1D.sum() == 0: continue
            
            if BC == 'periodic': #if boundary conditions are periodic, meaning updrafts continue to the other side past the boundary
                array1D = np.tile(array1D, 3)
                labeled, sizes = LabelSizesOfIndividualUpdrafts(array1D=array1D)
                lengthWhere = ApplyLengths(array1D=array1D, sizes=sizes, labeled=labeled)
                lengthWhere = np.minimum(lengthWhere[N:2*N], N)
            elif BC == 'open': #if boundary conditions are open (e.g. radiative), meaning updrafts disappear at the boundary
                labeled, sizes = LabelSizesOfIndividualUpdrafts(array1D=array1D)
                lengthWhere = ApplyLengths(array1D=array1D, sizes=sizes, labeled=labeled)

            if dimension == 'x':
                length[k, n, :] = lengthWhere
            elif dimension == 'y':
                length[k, :, n] = lengthWhere
    return length

# def CalculateBinaryArea_V2(A,w,
#                         shapeApproximation="oval"):
#     """
#     For each updraft point, calculate the local cross-sectional area
#     as xLength * yLength.

#     NOT RECOMMENDED SINCE UPDRAFT AREA COULD BE LARGE, BUT VERY CLOSE TO EDGE OF CLOUD
#     """
#     xLength = CalculateBinaryLength(A, dimension='x')
#     yLength = CalculateBinaryLength(A, dimension='y')

#     #calculating updraft area
#     updraftArea = xLength * yLength
#     #converting from grid index to meters
#     updraftArea *= ModelData.dx * ModelData.dy
#     #converting to oval approximation
#     if shapeApproximation=="oval":
#         updraftArea *= (np.pi / 4)
    
#     return updraftArea, xLength,yLength

def CalculateBinaryArea_V3(A,w, shapeApproximation="oval"):
    """
    For each updraft point, calculate area as mean_x_length * mean_y_length
    for the 2D connected region it belongs to.

    #RECOMMENDED
    """
    [Nz, Ny, Nx] = A.shape
    area = np.full(A.shape, np.nan)
    xLength_out = np.full(A.shape, np.nan)
    yLength_out = np.full(A.shape, np.nan)

    # Get per-point x and y lengths
    xLength = CalculateBinaryLength(A, dimension='x')
    yLength = CalculateBinaryLength(A, dimension='y')

    for k in range(Nz):
        slice2D = A[k, :, :]
        if slice2D.sum() == 0: continue

        # labeled, n_features = label(slice2D)
        labeled, n_features = GetLabeled_2D(slice2D,varSlice2D=w[k])
        

        for i in range(1, n_features + 1):
            mask = labeled == i
            xLength_out[k][mask] = np.nanmean(xLength[k][mask])
            yLength_out[k][mask] = np.nanmean(yLength[k][mask])

    #calculating updraft area
    updraftArea = xLength_out * yLength_out
    #converting from grid index to meters
    updraftArea *= ModelData.dx * ModelData.dy
    #converting to oval approximation
    if shapeApproximation == "oval":
        updraftArea *= (np.pi / 4)

    return updraftArea, xLength_out, yLength_out

from scipy.ndimage import distance_transform_edt
def CalculateEuclideanDistanceFromNearestEdge(A,dy=ModelData.dy,dx=ModelData.dx):
    """
    For each updraft point, calculate the Euclidean distance to the 
    nearest non-updraft cell.
    """
    [Nz, Ny, Nx] = A.shape
    distance = np.full(A.shape, np.nan)

    for k in range(Nz):
        slice2D = A[k, :, :]
        if slice2D.sum() == 0: continue
        # sampling gives physical distance in meters
        distance[k] = np.where(slice2D, distance_transform_edt(slice2D, sampling=(dy,dx)), np.nan)

    return distance

In [ ]:
#Running Functions

def VariableCalculation(inputDictionary,varNames):
    [A_g,A_c,w] = (inputDictionary[k] for k in varNames)

    [updraftArea_g,_,_] = CalculateBinaryArea_V3(A_g,w)
    [updraftArea_c,_,_] = CalculateBinaryArea_V3(A_c,w)

    updraftEdgeDistance_g = CalculateEuclideanDistanceFromNearestEdge(A_g)
    updraftEdgeDistance_c = CalculateEuclideanDistanceFromNearestEdge(A_c)
    
    outputDictionary={'updraftArea_g': updraftArea_g,
                      'updraftArea_c': updraftArea_c,
                      'updraftEdgeDistance_g': updraftEdgeDistance_g,
                      'updraftEdgeDistance_c': updraftEdgeDistance_c}
    
    return outputDictionary

def RunCode():
    for t in tqdm(loop_elements, total=len(loop_elements)):
        if np.mod(t,1)==0: print(f'Current time {t}')   

        #get variable names
        varNames = GetVarNames()
    
        #loading input variables
        timeString = ModelData.timeStrings[t]
        inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString, varNames)
    
        #calculating variables
        outputDictionary = VariableCalculation(inputDictionary,varNames)
        
        #outputting
        DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary)

In [ ]:
####################################
#RUNNING
running=True #Keep True When Running COode
# running=False

In [ ]:
#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
RunCode()

In [ ]:
####################################
#PLOTTING 

# Updraft Threshold Fields and Updraft Area Calculations look (Case Study)

In [ ]:
#Plotting Functions

def PlotSlice(data, mask, xLevel=100, yLevels=np.arange(0,200), varType='symmetric',
              yLim=None, numLevels=100,levels=None,zorder=None,
              axis=None,cbarLabel='w (m/s)'):

    if axis is None:
        fig,axis=plt.subplots()

    cmap='RdBu_r' if varType == 'symmetric' else 'turbo'

    #setting z limit
    zHeight=6
    zIndex = np.abs(ModelData.zh-zHeight).argmin()
    data_2=data.copy()
    data_2[zIndex+1:]=np.nan
    axis.set_ylim(0, zHeight) if yLim is None else axis.set_ylim(yLim)
    
    data_plot = np.where(mask[:, yLevels, xLevel], data_2[:, yLevels, xLevel], np.nan)

    if levels is None:
        if varType == 'symmetric':
            vmax=np.nanmax(data_plot) 
            levels = np.linspace(-vmax,vmax,numLevels)
        elif varType == 'positive':
            levels=numLevels



    cf=axis.contourf(ModelData.yh[yLevels],ModelData.zh,data_plot,cmap=cmap,levels=levels,zorder=zorder)
    plt.colorbar(cf, ax=axis,label=cbarLabel)

    axis.set_ylabel('z (km)'); axis.set_xlabel('y (km)'); 
    
    
def Plot_XYSlice(axis):
    #Finding Ideal Cloud
    
    #Simulation
    #Simulation 4
    
    #Height
    zHeight=2.01
    zIndex = np.abs(ModelData.zh-zHeight).argmin()
    
    #Cloud
    a=qcqi[zIndex].copy()
    # a[(a<=1e-6)]=np.nan
    cf=axis.contourf(ModelData.xh,ModelData.yh,a,cmap='turbo',levels=16,zorder=0);plt.colorbar(cf,ax=axis,label='qc+qi (kg/kg)')
    axis.contour(a>1e-6,colors='red',zorder=2)
    
    #Updraft
    a=w[zIndex].copy()
    a[(a<=0.1)&(a>=-0.1)]=np.nan
    axis.contour(ModelData.xh,ModelData.yh,a,colors='white',levels=16,zorder=3)
    
    #Refining XLim
    yLevels=np.arange(90,105+1)
    axis.set_ylim(yLevels[0],yLevels[-1])
    step=5
    axis.set_xlim(180-step,185+step)
    
    
    #==>
    xLevel=183
    yLevel=98
    axis.axvline(xLevel,color='black',linewidth=1)
    axis.axhline(yLevel,color='black',linewidth=1)
    
    axis.set_xlabel('x (km)'); axis.set_ylabel('y (km)')
    axis.set_title('qc+qi (contour): qc+qi > 1e-6 (red)\nw (lines): w>0.1 - and w<-0.1 -- (white)');

    return zHeight,zIndex,yLevels,yLevel,xLevel

def Plot_ZYSlice(axis):

    updraft = (w==w) #disabled
    PlotSlice(data=w,mask=updraft, xLevel=xLevel, yLevels=yLevels,varType='positive',zorder=0,axis=axis,cbarLabel='w (m/s)')
    cf=axis.contour(ModelData.yh[yLevels],ModelData.zh,qcqi[:,yLevels,xLevel]>1e-6,colors='red',zorder=2)
    
    axis.contour(ModelData.yh[yLevels],ModelData.zh,w[:,yLevels,xLevel]>0.1,colors='white',linewidths=1,zorder=3)
    axis.contour(ModelData.yh[yLevels],ModelData.zh,w[:,yLevels,xLevel]>0.5,colors='white',linestyles='dashed',linewidths=1,zorder=3)
    axis.axvline(yLevel,color='black',linewidth=2,zorder=1)
    axis.axhline(zHeight,color='black',linewidth=2,zorder=1)

    axis.set_title('w (contour): qc+qi > 1e-6 (red)\nw (lines): w>0.1 - and w>0.5 -- (white)');

def Plot_ZYSlice_VarData(axis,varData,cbarLabel="",levels=np.arange(0,10+1)):
    
    updraft = (varData==varData) #disabled
    PlotSlice(data=varData,mask=updraft, xLevel=xLevel, yLevels=yLevels,varType='positive',zorder=0,levels=levels,
              axis=axis,cbarLabel=cbarLabel)
    axis.contour(ModelData.yh[yLevels],ModelData.zh,qcqi[:,yLevels,xLevel]>1e-6,colors='red',zorder=2)
    
    axis.contour(ModelData.yh[yLevels],ModelData.zh,w[:,yLevels,xLevel]>0.1,colors='white',linewidths=1,zorder=3)
    axis.contour(ModelData.yh[yLevels],ModelData.zh,w[:,yLevels,xLevel]>0.5,colors='white',linestyles='dashed',
                 linewidths=1,zorder=3)
    axis.axvline(yLevel,color='black',linewidth=2,zorder=1)
    axis.axhline(zHeight,color='black',linewidth=2,zorder=1)

In [ ]:
#Calculating And Loading Data
if not running:
    
    t=200
    timeString = ModelData.timeStrings[t]
    
    #Calculate Updraft Area
    if "outputDictionary" not in globals():
        #get variable names
        varNames = GetVarNames()
        #loading input variables
        inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString, varNames)
        #calculating variables
        outputDictionary = VariableCalculation(inputDictionary,varNames)
        updraftEdgeDistance_g=outputDictionary['updraftEdgeDistance_g']/1e3
        updraftEdgeDistance_c=outputDictionary['updraftEdgeDistance_c']/1e3
        updraftArea_g=outputDictionary['updraftArea_g']
        updraftArea_c=outputDictionary['updraftArea_c']
        
        updraftArea_g=np.sqrt(updraftArea_g/1e6)
        updraftArea_c=np.sqrt(updraftArea_c/1e6)
    
        #making it so contourf works for plotting
        updraftArea_g[np.isnan(updraftArea_g)]=0 
        updraftArea_c[np.isnan(updraftArea_c)]=0
        updraftEdgeDistance_g[np.isnan(updraftEdgeDistance_g)]=0
        updraftEdgeDistance_c[np.isnan(updraftEdgeDistance_c)]=0
    
    #Calculate Other Variables
    varNames = GetVarNames()+['qcqi','buoyancy2']
    inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString, varNames)
        
    [A_g,A_c,w,qcqi,B] = (inputDictionary[k] for k in varNames)

In [ ]:
if not running:
    #Plotting
    fig,axes=plt.subplots(3,2,figsize=(16,12))
    axes=axes.ravel()
    
    [zHeight,zIndex,yLevels,yLevel,xLevel] = Plot_XYSlice(axis=axes[0])
    Plot_ZYSlice(axis=axes[1])
    Plot_ZYSlice_VarData(axes[2],updraftArea_g,levels=np.arange(0,10+1),cbarLabel="sqrt( updraftArea_g ) (km)")
    Plot_ZYSlice_VarData(axes[3],updraftArea_c,levels=np.arange(0,10+1),cbarLabel="sqrt( updraftArea_c ) (km)")
    Plot_ZYSlice_VarData(axes[4],updraftEdgeDistance_g,levels=15,cbarLabel="updraftEdgeDistance_g (km)")
    Plot_ZYSlice_VarData(axes[5],updraftEdgeDistance_c,levels=15,cbarLabel="updraftEdgeDistance_c (km)")